In [ ]:
#| default_exp clis

# CLI Tools

> Command-line interface for Planet Four catalog production.

Provides the `p4` entry point with subcommands for setup, inspection,
single-tile clustering, obsid-level clustering, and full catalog production.
Uses Rich for TUI output and ProcessPoolExecutor for parallelism.

## Installation

The `p4` CLI is installed automatically when you install `p4tools`:

```bash
pip install -e .
```

This registers the `p4` entry point (defined via `console_scripts` in `settings.ini`).
The CLI depends on [Typer](https://typer.tiangolo.com/) and [Rich](https://rich.readthedocs.io/).

## Commands Overview

| Command | Description |
|---------|-------------|
| `p4 setup` | Configure the Planet Four data root and database path |
| `p4 info` | Show current configuration and database summary |
| `p4 cluster-tile` | Cluster a single tile with optional inline plot |
| `p4 cluster-obsid` | Cluster all tiles for a HiRISE observation ID |
| `p4 produce` | Run full catalog production with parallel processing |

In [ ]:
#| export
from __future__ import annotations

import logging
import sys
import tempfile
import base64
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Optional

import typer
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.progress import (
    Progress, SpinnerColumn, TextColumn,
    BarColumn, MofNCompleteColumn, TimeElapsedColumn,
)

import p4tools.production.io as io

In [ ]:
#| export
app = typer.Typer(
    name="p4",
    help="Planet Four catalog production tools.",
    rich_markup_mode="rich",
    no_args_is_help=True,
)
console = Console()

## Parallel Execution Helper

`run_parallel_with_progress` is a general-purpose utility that wraps
`ProcessPoolExecutor` with a Rich progress bar. It is used by the
`produce` command to parallelize clustering and fnotching across
hundreds of observation IDs.

In [ ]:
#| export
def run_parallel_with_progress(
    func,
    items,
    max_workers: int = 4,
    description: str = "Processing",
    func_kwargs: dict | None = None,
):
    """Execute func(item, **func_kwargs) in parallel with a Rich progress bar.

    Parameters
    ----------
    func : callable
        A *picklable* top-level function.  Must accept a single positional
        argument (the item) plus any keyword arguments from *func_kwargs*.
    items : sequence
        Items to map over.
    max_workers : int
        Number of parallel processes.
    description : str
        Label shown in the progress bar.
    func_kwargs : dict, optional
        Extra keyword arguments forwarded to *func*.

    Returns
    -------
    results : list
        Successful results in the original item order (skipping failures).
    errors : list[tuple]
        ``(item, exception)`` pairs for items that raised.
    """
    if func_kwargs is None:
        func_kwargs = {}
    results, errors = {}, []
    items = list(items)
    with Progress(
        SpinnerColumn(),
        TextColumn("[bold blue]{task.description}"),
        BarColumn(),
        MofNCompleteColumn(),
        TimeElapsedColumn(),
    ) as progress:
        task = progress.add_task(description, total=len(items))
        with ProcessPoolExecutor(max_workers=max_workers) as executor:
            future_to_item = {
                executor.submit(func, item, **func_kwargs): item
                for item in items
            }
            for future in as_completed(future_to_item):
                item = future_to_item[future]
                try:
                    results[item] = future.result()
                except Exception as e:
                    errors.append((item, e))
                    progress.console.print(f"[red]Error {item}: {e}[/red]")
                progress.advance(task)
    return [results[item] for item in items if item in results], errors

## `p4 setup`

Configures the `~/.p4tools.ini` file with the path to the data root
directory where clustering results are stored. Optionally also stores
the path to the raw Planet Four Parquet database.

```bash
p4 setup ~/planet4_data --db ~/planet4_data/p4_raw.parquet
```

This replaces the old interactive prompt that ran at import time
when no config file existed.

In [ ]:
#| export
@app.command()
def setup(
    data_root: str = typer.Argument(help="Path where Planet Four results are stored."),
    db: Optional[str] = typer.Option(None, "--db", help="Path to raw P4 Parquet database."),
):
    """Configure the Planet Four data root (and optionally the raw database path)."""
    data_root_path = Path(data_root).expanduser().resolve()
    data_root_path.mkdir(parents=True, exist_ok=True)
    io.set_database_path(str(data_root_path))

    if db is not None:
        import configparser
        config = configparser.ConfigParser()
        config.read(str(io.configpath))
        if "planet4_db" not in config:
            config["planet4_db"] = {}
        config["planet4_db"]["dbname"] = str(Path(db).expanduser().resolve())
        with io.configpath.open("w") as f:
            config.write(f)

    console.print(Panel(
        f"[green]Data root:[/green] {data_root_path}\n"
        + (f"[green]Database:[/green]  {db}" if db else "[dim]No database path set.[/dim]"),
        title="Planet Four Configuration",
    ))

## `p4 info`

Displays the current configuration in a Rich table: config file
location, data root path, and all key/value pairs from the INI file.

```bash
p4 info
```

In [ ]:
#| export
@app.command()
def info():
    """Show current configuration and database summary."""
    table = Table(title="Planet Four Configuration")
    table.add_column("Key", style="cyan")
    table.add_column("Value", style="white")

    table.add_row("Config file", str(io.configpath))
    table.add_row("Config exists", str(io.configpath.exists()))

    if io.configpath.exists():
        config = io.get_config()
        for section in config.sections():
            for key, val in config[section].items():
                table.add_row(f"{section}.{key}", val)

        if io.data_root is not None:
            table.add_row("Data root", str(io.data_root))
            table.add_row("Data root exists", str(io.data_root.exists()))
    else:
        table.add_row("[yellow]Status[/yellow]", "[yellow]Not configured. Run 'p4 setup'.[/yellow]")

    console.print(table)

## Terminal Image Display

`_display_inline_image` renders a PNG/JPEG inline in the terminal
using the [iTerm2 inline image protocol](https://iterm2.com/documentation-images.html)
(also supported by WezTerm). Falls back to printing the file path
on unsupported terminals. Used by `cluster-tile` to show plots
without leaving the CLI.

In [ ]:
#| export
def _display_inline_image(path: Path):
    """Display an image inline in the terminal using iTerm2 protocol.

    Falls back to printing the file path if the terminal does not support
    the iTerm2 inline image protocol.

    Parameters
    ----------
    path : Path
        Path to an image file (PNG/JPEG).
    """
    import os
    term_program = os.environ.get("TERM_PROGRAM", "")
    if term_program in ("iTerm.app", "WezTerm"):
        data = path.read_bytes()
        b64 = base64.b64encode(data).decode("ascii")
        sys.stdout.write(f"\033]1337;File=inline=1;size={len(data)}:{b64}\a")
        sys.stdout.write("\n")
        sys.stdout.flush()
    else:
        console.print(f"[dim]Plot saved to:[/dim] {path}")

## `p4 cluster-tile`

Clusters citizen science markings on a single Planet Four tile using
DBSCAN, then displays a summary table with fan/blotch cluster counts.
With `--plot` (the default), it also renders a 3-panel figure showing
the tile image, clustered fans, and clustered blotches.

```bash
# Cluster a single tile
p4 cluster-tile APF00003qk --db ~/planet4_data/p4_raw.parquet

# Without the inline plot
p4 cluster-tile APF00003qk --db ~/planet4_data/p4_raw.parquet --no-plot
```

In [ ]:
#| export
@app.command()
def cluster_tile(
    tile_id: str = typer.Argument(help="Planet Four tile image_id (e.g. APF00003qk)."),
    db: str = typer.Option(..., "--db", help="Path to raw P4 Parquet database."),
    savedir: Optional[str] = typer.Option(None, "--savedir", help="Directory for clustering output. Defaults to data_root/clustering."),
    plot: bool = typer.Option(True, help="Show an inline plot of the clustered markings."),
):
    """Cluster a single Planet Four tile and display the results."""
    from p4tools.production import markings, dbscan

    tile_id = io.check_and_pad_id(tile_id)
    console.print(f"Clustering tile [cyan]{tile_id}[/cyan] ...")

    # resolve obsid
    db_mgr = io.DBManager(dbname=db)
    obsid = db_mgr.get_obsid_for_tile_id(tile_id)
    console.print(f"  Obsid: [cyan]{obsid}[/cyan]")

    # cluster
    effective_savedir = savedir or "clustering"
    scanner = dbscan.DBScanner(savedir=effective_savedir, dbname=db)
    scanner.pm.obsid = obsid
    scanner.pm.id = tile_id
    scanner.cluster_image_id(tile_id, image_name=obsid)

    # read results
    pm = io.PathManager(id_=tile_id, obsid=obsid, datapath=effective_savedir)

    table = Table(title=f"Clustering Results for {tile_id}")
    table.add_column("Marking", style="cyan")
    table.add_column("Clusters", justify="right")
    table.add_column("File", style="dim")

    for kind, fpath in [("fans", pm.fanfile), ("blotches", pm.blotchfile)]:
        if fpath.exists():
            import pandas as pd
            df = pd.read_csv(fpath)
            table.add_row(kind, str(len(df)), str(fpath.name))
        else:
            table.add_row(kind, "0", "[dim]no file[/dim]")

    console.print(table)

    # inline plot
    if plot:
        try:
            import matplotlib
            matplotlib.use("Agg")
            import matplotlib.pyplot as plt

            p4id = markings.TileID(tile_id, dbname=db)
            fig, axes = plt.subplots(1, 3, figsize=(18, 6))

            # original markings
            p4id.show_subframe(ax=axes[0])
            axes[0].set_title("Tile image")

            # clustered fans
            p4id.show_subframe(ax=axes[1])
            if pm.fanfile.exists():
                import pandas as pd
                fan_df = pd.read_csv(pm.fanfile)
                p4id.plot_fans(data=fan_df, ax=axes[1], lw=1, with_center=True)
            axes[1].set_title(f"Fans ({len(fan_df) if pm.fanfile.exists() else 0})")

            # clustered blotches
            p4id.show_subframe(ax=axes[2])
            if pm.blotchfile.exists():
                import pandas as pd
                blotch_df = pd.read_csv(pm.blotchfile)
                p4id.plot_blotches(data=blotch_df, ax=axes[2], lw=1, with_center=True)
            axes[2].set_title(f"Blotches ({len(blotch_df) if pm.blotchfile.exists() else 0})")

            fig.suptitle(f"{tile_id} — {obsid}", fontsize=14)
            fig.tight_layout()

            with tempfile.NamedTemporaryFile(suffix=".png", delete=False) as tmp:
                fig.savefig(tmp.name, dpi=120, bbox_inches="tight")
                plt.close(fig)
                _display_inline_image(Path(tmp.name))
        except Exception as e:
            console.print(f"[yellow]Could not render plot: {e}[/yellow]")

## `p4 cluster-obsid`

Clusters all tiles belonging to a HiRISE observation. This is the
obsid-level equivalent of `cluster-tile`. Optionally runs fnotching
(fan/blotch ambiguity resolution) after clustering.

```bash
# Cluster only
p4 cluster-obsid ESP_011296_0975 --db ~/planet4_data/p4_raw.parquet

# Cluster + fnotch
p4 cluster-obsid ESP_011296_0975 --db ~/planet4_data/p4_raw.parquet --fnotch
```

In [ ]:
#| export
@app.command()
def cluster_obsid(
    obsid: str = typer.Argument(help="HiRISE observation ID (image_name)."),
    db: str = typer.Option(..., "--db", help="Path to raw P4 Parquet database."),
    savedir: Optional[str] = typer.Option(None, "--savedir", help="Output directory. Defaults to data_root/clustering."),
    fnotch: bool = typer.Option(False, "--fnotch/--no-fnotch", help="Also run fnotching after clustering."),
):
    """Cluster all tiles for a HiRISE observation ID."""
    from p4tools.production.catalog import cluster_obsid as _cluster_obsid, fnotch_obsid as _fnotch_obsid

    effective_savedir = savedir or "clustering"

    with console.status(f"Clustering [cyan]{obsid}[/cyan] ..."):
        _cluster_obsid(obsid=obsid, savedir=effective_savedir, dbname=db)

    console.print(f"[green]Clustering complete for {obsid}.[/green]")

    if fnotch:
        with console.status(f"Fnotching [cyan]{obsid}[/cyan] ..."):
            _fnotch_obsid(obsid=obsid, savedir=effective_savedir)
        console.print(f"[green]Fnotching complete for {obsid}.[/green]")

    # summary table
    pm = io.PathManager(obsid=obsid, datapath=effective_savedir)
    l1a_paths = pm.get_obsid_paths("L1A")

    table = Table(title=f"Results for {obsid}")
    table.add_column("Metric", style="cyan")
    table.add_column("Value", justify="right")
    table.add_row("Tiles processed", str(len(l1a_paths)))

    console.print(table)

### Picklable Wrappers

`_cluster_single` and `_fnotch_single` are top-level wrapper functions
that can be pickled by `ProcessPoolExecutor`. They perform lazy imports
to avoid serialization issues with module-level state.

In [ ]:
#| export
# Top-level picklable wrappers for ProcessPoolExecutor

def _cluster_single(obsid, savedir=None, dbname=None, min_cluster_size=3):
    """Picklable wrapper around catalog.cluster_obsid for parallel execution."""
    from p4tools.production.catalog import cluster_obsid as _cluster_obsid
    return _cluster_obsid(
        obsid=obsid, savedir=savedir, dbname=dbname,
        min_cluster_size=min_cluster_size,
    )


def _fnotch_single(obsid, savedir=None):
    """Picklable wrapper around catalog.fnotch_obsid for parallel execution."""
    from p4tools.production.catalog import fnotch_obsid as _fnotch_obsid
    return _fnotch_obsid(obsid=obsid, savedir=savedir)

## `p4 produce`

The main catalog production command. Runs the full pipeline in four
phases:

1. **Clustering** — Parallel DBSCAN clustering of all observation IDs
2. **Fnotching** — Parallel fan/blotch ambiguity resolution
3. **Post-processing** — L1C summaries, tile/marking coordinates, metadata
4. **Marking IDs** — Assigns unique identifiers to all catalog entries

Uses `run_parallel_with_progress` for phases 1 and 2, with configurable
worker count. Supports `--dry-run` to preview the work without executing.

```bash
# Full production run
p4 produce v3.1 --db ~/planet4_data/p4_raw.parquet --workers 8

# Preview what would be done
p4 produce v3.1 --db ~/planet4_data/p4_raw.parquet --dry-run
```

In [ ]:
#| export
@app.command()
def produce(
    version: str = typer.Argument(help="Catalog version string (e.g. v1.0)."),
    db: str = typer.Option(..., "--db", help="Path to raw P4 Parquet database."),
    workers: int = typer.Option(4, "--workers", "-w", help="Number of parallel workers."),
    dry_run: bool = typer.Option(False, "--dry-run", help="Show what would be done without executing."),
):
    """Run full catalog production with parallel processing and Rich progress."""
    from p4tools.production.catalog import (
        ReleaseManager, get_L1A_paths, add_marking_ids,
        fan_id_generator, blotch_id_generator, create_roi_file,
    )

    rm = ReleaseManager(version=version, dbname=db)
    rm.check_for_todo()
    obsids = rm.todo

    console.print(Panel(
        f"[cyan]Version:[/cyan]  {version}\n"
        f"[cyan]Database:[/cyan] {db}\n"
        f"[cyan]Workers:[/cyan]  {workers}\n"
        f"[cyan]Obsids:[/cyan]   {len(obsids)} to process\n"
        f"[cyan]Save to:[/cyan]  {rm.savefolder}",
        title="Catalog Production Plan",
    ))

    if dry_run:
        console.print("[yellow]Dry run — no work performed.[/yellow]")
        if len(obsids) <= 20:
            for obs in obsids:
                console.print(f"  {obs}")
        return

    if len(obsids) == 0:
        console.print("[green]Nothing to do — all obsids already processed.[/green]")
        return

    # --- Phase 1: Clustering (parallel) ---
    console.rule("[bold]Phase 1: Clustering[/bold]")
    _, cluster_errors = run_parallel_with_progress(
        _cluster_single, obsids, max_workers=workers,
        description="Clustering obsids",
        func_kwargs={"savedir": version, "dbname": db},
    )
    if cluster_errors:
        console.print(f"[red]{len(cluster_errors)} obsids failed clustering.[/red]")

    # --- Phase 2: Fnotching (parallel) ---
    console.rule("[bold]Phase 2: Fnotching[/bold]")
    _, fnotch_errors = run_parallel_with_progress(
        _fnotch_single, obsids, max_workers=workers,
        description="Fnotching obsids",
        func_kwargs={"savedir": version},
    )
    if fnotch_errors:
        console.print(f"[red]{len(fnotch_errors)} obsids failed fnotching.[/red]")

    # --- Phase 3: Post-processing (serial) ---
    console.rule("[bold]Phase 3: Post-processing[/bold]")

    with console.status("Creating L1C summary files ..."):
        create_roi_file(rm.obsids, rm.catalog, version)
    console.print("  [green]L1C summary files created.[/green]")

    with console.status("Calculating tile coordinates ..."):
        rm.calc_tile_coordinates()
    console.print("  [green]Tile coordinates calculated.[/green]")

    with console.status("Calculating marking coordinates ..."):
        rm.calc_marking_coordinates()
    console.print("  [green]Marking coordinates calculated.[/green]")

    with console.status("Writing metadata ..."):
        rm.calc_metadata()
    console.print("  [green]Metadata written.[/green]")

    with console.status("Merging all catalog data ..."):
        rm.merge_all()
    console.print("  [green]Catalog merged.[/green]")

    # --- Phase 4: Marking IDs (serial, final step) ---
    console.rule("[bold]Phase 4: Assigning Marking IDs[/bold]")
    with console.status("Assigning unique marking IDs ..."):
        rm.fix_marking_ids()
    console.print("  [green]Marking IDs assigned.[/green]")

    console.print(Panel(
        f"[green]Catalog production complete![/green]\n"
        f"Output: {rm.savefolder}",
        title="Done",
    ))

In [ ]:
#| export
#| eval: false
if __name__ == "__main__":
    app()